# CodeAlpha Data Analytics Internship - Project 2: Data Visualization
## FIFA World Cup 2026 Player Performance & Valuation Analytics

**Author:** Data Analytics Intern  
**Repository:** CodeAlpha_Data-Visualization  

---

## 1. Project Overview & Objectives
This project focuses on exploratory data visualization and analytics using a dataset detailing player performances leading up to the **FIFA World Cup 2026**.

### Objectives:
- Gain high-level demographical insights (age, positions, foot preference).
- Explore player market values and overall performance ratings.
- Clean the dataset and analyze key correlations and distribution shapes.
- Propose actionable business recommendations and visual dashboard design.

### Step 1: Import Libraries and Setup Plots

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style setup
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight'
})

# Ensure output image directory exists
os.makedirs('../images/generated_visualizations', exist_ok=True)

### Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('../data/dataset.csv')
print(f"Dataset Shape: {df.shape}")

### Step 3: Explore First & Last Records

In [ ]:
df.head()

### Step 4: Summary Statistics & Feature Information

In [ ]:
df.info()
display(df.describe())

### Step 5: Check Missing Values & Duplicates

In [ ]:
print("Missing Values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDuplicate Records: {df.duplicated().sum()}")

## 2. Data Cleaning & Preprocessing

We apply the following corrections:
- Convert `match_date` to Datetime objects.
- Remove duplicates if present.
- Fill missing values logically (e.g. Goalkeeper stats like `saves` to 0 for outfield players).

In [ ]:
# 1. Convert Date
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')

# 2. Remove Duplicates
df = df.drop_duplicates()

# 3. Standardize Object fields
for col in ['preferred_foot', 'position']:
    df[col] = df[col].astype(str).str.strip().str.title()

# 4. Impute Goalkeeper specific missing values
gk_cols = ['saves', 'save_percentage', 'punches', 'penalty_saves', 'goals_conceded']
for col in gk_cols:
    df[col] = df[col].fillna(0)

print(f"Preprocessed Dataset Shape: {df.shape}")

## 3. Data Visualization & Insights

### 1. Age Distribution of Players
**Objective:** Visualize squad age distributions to identify peak physical development profiles.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='age', kde=True, color='#0D9488', bins=18)
plt.title('Player Age Distribution (FIFA World Cup 2026)', weight='bold')
plt.xlabel('Age (Years)')
plt.ylabel('Count')
plt.savefig('../images/generated_visualizations/age_distribution.png')
plt.show()

**What this chart shows:** The histogram and KDE fit show player age concentrations.

**Observation:** Player counts peak significantly in the 23-27 age window, showing that national teams heavily rely on players in their physical prime.

**Business Insight:** Recruitment networks should lock in players around age 21-22 before they enter this peak value bracket and experience high valuations.

### 2. Player Positions Count
**Objective:** Understand squad depth and representation across different positions.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=df.groupby('player_id').first().reset_index(), y='position', order=df['position'].value_counts().index, palette='crest')
plt.title('Player Distribution by Position', weight='bold')
plt.xlabel('Count')
plt.ylabel('Position')
plt.savefig('../images/generated_visualizations/position_distribution.png')
plt.show()

**What this chart shows:** A counts summary of distinct positions.

**Observation:** Midfielders and Defenders dominate squad sizes, while Goalkeepers and Forwards are fewer in number.

**Business Insight:** The talent pool for elite strikers is small, increasing their price premiums during transfer windows.

### 3. Market Value by Position
**Objective:** Evaluate valuation distributions across player positions.

In [ ]:
plt.figure(figsize=(10, 6))
df_unique = df.groupby('player_id').first().reset_index()
df_unique['market_value_m'] = df_unique['market_value_eur'] / 1e6
sns.boxplot(data=df_unique, x='position', y='market_value_m', palette='Set2')
plt.yscale('log')
plt.title('Market Value Distribution by Position (Log Scale)', weight='bold')
plt.xlabel('Position')
plt.ylabel('Market Value (Million EUR)')
plt.savefig('../images/generated_visualizations/market_value_by_position.png')
plt.show()

**What this chart shows:** Outlier-inclusive box plot showing financial spreads.

**Observation:** Forwards and Midfielders command much higher median market valuations, showing a premium placed on game-winners.

**Business Insight:** Allocating financial resources towards elite forwards is high risk/high reward; building strong internal youth academies for defensive roles is a smarter long-term strategy.

### 4. Player Rating vs. Market Value
**Objective:** Find the performance-to-value correlation structure.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df_unique, x='player_rating', y='market_value_m', hue='position', alpha=0.6)
plt.title('Tournament Rating vs. Market Value', weight='bold')
plt.xlabel('Player Rating')
plt.ylabel('Market Value (Million EUR)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig('../images/generated_visualizations/rating_vs_value.png')
plt.show()

**What this chart shows:** A scatter plot representing market values against actual overall tournament ratings.

**Observation:** Valuations remain low and linear until ratings reach 8.0, where value increases exponentially.

**Business Insight:** Clubs should target players with ratings between 7.5 and 8.0, who offer high performance before entering the high-cost superstar tier.

### 5. Key Metrics Correlation Heatmap
**Objective:** Correlate numerical features to identify the primary performance metrics driving valuations.

In [ ]:
plt.figure(figsize=(10, 8))
cols = ['age', 'market_value_eur', 'player_rating', 'goals', 'assists', 'key_passes', 'pass_accuracy', 'stamina_score']
sns.heatmap(df[cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True, linewidths=0.5)
plt.title('Performance & Financial Metrics Correlation Matrix', weight='bold')
plt.savefig('../images/generated_visualizations/correlation_heatmap.png')
plt.show()

**What this chart shows:** A correlation matrix heatmap.

**Observation:** Goals, assists, and player rating are highly correlated with market value. Stamina and physical metrics show low direct correlation with market price.

**Business Insight:** Creative and scoring outputs dictate financial value, meaning clubs can acquire physically superior, high-workrate players for lower prices.

## 4. Key Recommendations
1. **Buy Low in 7.5-8.0 Band:** Players inside this window offer high outputs without the premium superstar tag.
2. **Value High Stamina:** Purchase physically robust players for pressing systems cheaply, since physical attributes do not inflate market value metrics as much as offensive ones.
3. **Invest in Youth Forwards:** Given the low strikers count and high market valuations, prioritize forward academy prospects.